In [7]:
import os
import pandas as pd
import numpy as np
import bayesfit as bf
from IPython.display import clear_output

In [16]:
def parse_txt(full_file):
    filepath = '../data/0-raw/' + full_file
    filename,filetype = full_file.split('.')
    monkey=filename[0]
    if monkey=='t':
        monkey='Y'
    date=filename[1:]
    data = []
    row = {}
    with open(filepath) as file:
        line = file.readline()
        i=1
        while line:
            if line.strip():
                key, value = line.split(maxsplit=1)
                value = value.strip()
                if key=='Task':
                    if row:
                        row['Monkey'] = monkey
                        row['Date'] = int(date)
                        data.append(row)
                    row={key:value}
                elif row:
                    row[key] = value
            line = file.readline()
    return pd.DataFrame(data)

In [17]:
# load list of raw data files
files = os.listdir('../data/0-raw')
files.remove('.ipynb_checkpoints')

ValueError: list.remove(x): x not in list

In [40]:
i=0
clean_sessions = []
sessions = []
for f in files:
    clear_output(wait=True)
    print(f)
    filename,filetype = f.split('.')
    monkey=f[0]
    if monkey=='t':
        monkey='Y'
    date=filename[1:]
    sessions.append({'monkey': monkey, 'date': date, 'filename':f})
    session_data = parse_txt(f)
    session_clean = pd.DataFrame()
    if 'Active1' in session_data.columns:
        active_columns = [f'Active{x}' for x in range(1,9)]
        return_columns = [f'Return{x}' for x in range(1,9)]
        session_clean['Active Channels'] = session_data[active_columns].apply(lambda row: int(''.join(row.values), 2), axis=1)
        session_clean['Return Channels'] = session_data[return_columns].apply(lambda row: int(''.join(row.values), 2), axis=1)
    session_clean[['Reaction Time','Movement Time','Monkey','Date','Result']] = session_data[['Reaction_Time','Movement_Time','Monkey','Date','Result']]
    comp_columns = ['Comp Amp', 'Comp PW', 'Comp Freq', 'Comp Dur']
    ref_columns = ['Ref Amp', 'Ref PW', 'Ref Freq', 'Ref Dur']
    if 'Amp' in session_data.columns:
        session_clean[comp_columns] = session_data[['Amp','Width','Freq','Dur']].apply(pd.to_numeric, errors='coerce')
        session_clean = session_clean.assign(**dict.fromkeys(ref_columns,0))
    else:
        session_clean[comp_columns] = session_data[['Amp1','Width1','Freq1','Dur1']].apply(pd.to_numeric, errors='coerce')
        session_clean[ref_columns] = session_data[['Amp2','Width2','Freq2','Dur2']].apply(pd.to_numeric, errors='coerce')
    
    clean_sessions.append(session_clean)
    i = i+1
    print(f'{i} out of {len(files)} files completed')
# output: clean_sessions, a list of dataframes where each element of the list is a dataframe

t20170814.txt
433 out of 433 files completed


In [47]:
sessions = pd.DataFrame(sessions)
sessions.index = sessions.index.rename('id')

In [52]:
sessions.to_csv('../data/1-normalized/sessions.csv')

In [83]:
# pd.DataFrame(sessions).to_csv('data/tidy_data/sessions.csv')

In [22]:
sessions = clean_sessions.copy()
curveID = 1
sessions_w_curveIDs=[]
bad_sessions = []
for sess in sessions:
    sess_copy = sess.copy()
    print(curveID)
    clear_output(wait=True)
    try:
        params = sess_copy[['Ref Amp','Ref PW','Ref Freq','Ref Dur','Active Channels','Return Channels','Date']]
    except:
        bad_sessions.append(sess)
        continue
    diff = params.diff()
    next_curve = diff.any(axis='columns')
    for row, series in sess_copy.iterrows():
        if row:
            curveID = curveID + int(next_curve[row])
        else:
            curveID = curveID
        sess_copy.loc[row,'curveID'] = curveID
    curveID = curveID + 1
    sessions_w_curveIDs.append(sess_copy)

1105


In [23]:
chans={'Y20160901':['10000000','00001000'],
      'Y20160902':['00000100','01000000'],
      'Y20160906':['00000100','01000000'],
      'Y20160907':['00000100','01000000'],
      'Y20160908':['00000100','01000000'],
      'Y20160909':['00000100','01000000'],
      'Y20160912':['00001000','10000000'],
      'Y20160920':['00001111','10000000'],
      'Y20160921':['00001111','01000000'],
      'Y20160922':['00001111','00100000'],
      'Y20160926':['00001111','00100000'],
      'Y20160927':['00001111','00010000'],
      'U20160926':['00001000','10000000'],
      'U20161003':['00001111','01000000'],
      'U20160929':['00001111','10000000'],
      'U20160927':['00001001','10010000'],
      'U20160928':['00001000','10000000'],
       'U20161004':['10110000','01000000']
       }

i=1
for s in bad_sessions:
    session_str = s.loc[0,'Monkey'] + str(s.loc[0,'Date'])
    s['Active Channels'] = int(chans[session_str][0],2)
    s['Return Channels'] = int(chans[session_str][1],2)
    s['curveID'] = 1105 + i
    i=i+1
# copy[0]['ActiveChans'] = int('01110000',2)
# copy[0]['ReturnChans'] = int('10000000',2)

In [24]:
sessions_w_curveIDs.extend(bad_sessions)

In [25]:
sessions_w_curveIDs[0]

,Active Channels,Return Channels,Reaction Time,Movement Time,Monkey,Date,Result,Comp Amp,Comp PW,Comp Freq,Comp Dur,Ref Amp,Ref PW,Ref Freq,Ref Dur,curveID
0,240,4,-1052689,0,U,20161005,0,60,200,50,200,0,0,0,0,1.0
1,240,4,1019,442,U,20161005,1,1300,200,50,500,0,0,0,0,1.0
2,240,4,1019,442,U,20161005,3,1300,200,50,500,0,0,0,0,1.0
3,240,4,1019,442,U,20161005,3,1300,200,50,500,0,0,0,0,1.0
4,240,4,1199,474,U,20161005,1,1100,200,50,500,0,0,0,0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
534,240,4,1246,0,U,20161005,3,500,200,50,500,0,0,0,0,1.0
535,240,4,1246,0,U,20161005,3,500,200,50,500,0,0,0,0,1.0
536,240,4,1406,433,U,20161005,1,1100,200,50,500,0,0,0,0,1.0
537,240,4,1406,0,U,20161005,0,500,200,50,500,0,0,0,0,1.0


In [26]:
sessions = sessions_w_curveIDs.copy()
points_list = []
curves_list = []
for session in sessions:
    curves = session[['curveID','Ref Amp','Ref PW','Active Channels','Return Channels','Monkey','Date']].drop_duplicates().set_index('curveID')
    groups = session.groupby(['curveID','Comp Amp','Comp PW'])['Result'].value_counts()
    session_points = groups.unstack(fill_value=0)
    points_list.append(session_points)
    curves_list.append(curves)
points_table = pd.concat(points_list)
tidy_curves = pd.concat(curves_list)

In [27]:
points_table = points_table.rename(columns={'1':'Hits','0':'Misses','2':'FAs','3':'CRs'})
points_table = points_table[(points_table['Misses'] + points_table['Hits']) > 5]

In [28]:
points_table = points_table.reset_index()
points_table.index.name = 'pointID'

In [29]:
tidy_curves = tidy_curves.reset_index()

In [30]:
tidy_curves = tidy_curves[tidy_curves['curveID'].isin(points_table['curveID'])]

In [31]:
points_table.reset_index()
points_table.index.name = 'pointsID'
# points_table.to_csv('data/tidy_data/points.csv')

In [33]:
curves = tidy_curves
points = points_table.reset_index()

In [37]:
points

,pointsID,curveID,Comp Amp,Comp PW,Misses,Hits,FAs,CRs
0,0,1.0,300,200,28,6.0,3.0,48
1,1,1.0,500,200,26,8.0,3.0,38
2,2,1.0,700,200,14,20.0,3.0,28
3,3,1.0,900,200,0,34.0,2.0,24
4,4,1.0,1100,200,2,32.0,3.0,27
...,...,...,...,...,...,...,...,...
7477,7477,1122.0,60,200,17,25.0,30.0,22
7478,7478,1122.0,70,200,17,24.0,32.0,16
7479,7479,1122.0,80,200,8,35.0,38.0,21
7480,7480,1122.0,90,200,2,40.0,30.0,27


In [34]:
agg = curves_points.groupby('curveID')[['FAs','CRs']].sum()
curves = curves.merge(agg, right_index=True,left_on='curveID')

NameError: name 'curves_points' is not defined

In [69]:
points = points.drop(columns=['FAs','CRs'])

,pointID,curveID,Comp Amp,Comp PW,Misses,Hits,HR,p,d_prime
0,0,1.0,300,200,28,6.0,0.176471,0.125000,0.635827
1,1,1.0,500,200,26,8.0,0.235294,0.174923,0.731054
2,2,1.0,700,200,14,20.0,0.588235,0.544118,1.523161
3,3,1.0,900,200,0,34.0,1.000000,1.000000,3.604000
4,4,1.0,1100,200,2,32.0,0.941176,0.934641,2.846278
...,...,...,...,...,...,...,...,...,...
7477,7477,1122.0,60,200,17,25.0,0.111111,-0.939394,-1.325274
7478,7478,1122.0,70,200,17,24.0,0.068966,-0.862069,-1.483540
7479,7479,1122.0,80,200,8,35.0,0.705882,0.558824,0.972122
7480,7480,1122.0,90,200,2,40.0,0.882353,0.728507,1.018937


In [71]:
curves_points = curves.merge(points,on='curveID')

In [72]:
params = pd.DataFrame()
params['n_correct_signal'] = curves_points['Hits']
params['n_total_signal'] = curves_points['Hits'] + curves_points['Misses']
params['n_correct_noise'] = curves_points['CRs']
params['n_total_noise'] = curves_points['CRs'] + curves_points['FAs']

curves_points['HR'] = params['n_correct_signal'] / params['n_total_signal']
false_alarm_rate = 1 - (params['n_correct_noise'] / params['n_total_noise'])
curves_points['p'] = (curves_points['HR'] - false_alarm_rate) / (1 - false_alarm_rate)
curves_points['d_prime'] = params.apply(lambda x: dprime_yes_no_from_counts(
    x['n_correct_signal'], 
    x['n_total_signal'], 
    x['n_correct_noise'], 
    x['n_total_noise'],
    corr=True
),axis=1)
curves_points

/home/tyler/.pyenv/versions/3.7.4/lib/python3.7/site-packages/ipykernel_launcher.py:499: RuntimeWarning: divide by zero encountered in double_scalars


,curveID,Ref Amp,Ref PW,Active Channels,Return Channels,Monkey,Date,FAs,CRs,pointID,Comp Amp,Comp PW,Misses,Hits,HR,p,d_prime
0,1.0,0,0,240,4,U,20161005,24.0,245,0,300,200,28,6.0,0.176471,0.095798,0.416678
1,1.0,0,0,240,4,U,20161005,24.0,245,1,500,200,26,8.0,0.235294,0.160384,0.624056
2,1.0,0,0,240,4,U,20161005,24.0,245,2,700,200,14,20.0,0.588235,0.547899,1.568586
3,1.0,0,0,240,4,U,20161005,24.0,245,3,900,200,0,34.0,1.000000,1.000000,3.523501
4,1.0,0,0,240,4,U,20161005,24.0,245,4,1100,200,2,32.0,0.941176,0.935414,2.910304
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7517,1122.0,0,0,15,16,Y,20160927,273.0,152,7477,60,200,17,25.0,0.595238,-0.131736,-0.123715
7518,1122.0,0,0,15,16,Y,20160927,273.0,152,7478,70,200,17,24.0,0.585366,-0.159339,-0.149115
7519,1122.0,0,0,15,16,Y,20160927,273.0,152,7479,80,200,8,35.0,0.813953,0.479804,0.527804
7520,1122.0,0,0,15,16,Y,20160927,273.0,152,7480,90,200,2,40.0,0.952381,0.866855,1.303636


In [73]:
points[['HR','p','d_prime']] = curves_points[['HR','p','d_prime']]
points

,pointID,curveID,Comp Amp,Comp PW,Misses,Hits,HR,p,d_prime
0,0,1.0,300,200,28,6.0,0.176471,0.095798,0.416678
1,1,1.0,500,200,26,8.0,0.235294,0.160384,0.624056
2,2,1.0,700,200,14,20.0,0.588235,0.547899,1.568586
3,3,1.0,900,200,0,34.0,1.000000,1.000000,3.523501
4,4,1.0,1100,200,2,32.0,0.941176,0.935414,2.910304
...,...,...,...,...,...,...,...,...,...
7477,7477,1122.0,60,200,17,25.0,0.111111,-0.585687,-1.068226
7478,7478,1122.0,70,200,17,24.0,0.068966,-0.660871,-1.331125
7479,7479,1122.0,80,200,8,35.0,0.705882,0.475324,0.693810
7480,7480,1122.0,90,200,2,40.0,0.882353,0.790130,1.339246


In [74]:
unique_params = points.groupby('curveID')[['Comp Amp','Comp PW']].nunique()
unique_params = unique_params.rename(columns={'Comp Amp':'n_Amp','Comp PW':'n_PW'})
curves = curves.merge(unique_params,on='curveID')

In [77]:
# convert dates from str to datetime
curves['Date'] = pd.to_datetime(curves['Date'].apply(str))
# create column for days from start
start_dates = {'U':pd.to_datetime('2016-09-26'), 'Y':pd.to_datetime('2016-09-01'), 'Z': pd.to_datetime('2016-12-12')}
start_dates_series = curves['Monkey'].map(start_dates)
curves['Days'] = curves['Date'] - start_dates_series
curves['Days'] = curves['Days'].dt.days

,curveID,Ref Amp,Ref PW,Active Channels,Return Channels,Monkey,Date,FAs,CRs,n_Amp,n_PW,Days
0,1.0,0,0,240,4,U,2016-10-05,24.0,245,8,1,9
1,4.0,0,0,240,2,U,2016-10-06,17.0,280,8,1,10
2,5.0,0,0,15,128,U,2016-10-07,8.0,86,8,1,11
3,6.0,0,0,15,16,U,2016-10-07,5.0,199,8,1,11
4,7.0,0,0,15,240,U,2016-10-10,25.0,320,8,1,14


In [80]:
curves = curves[(curves['FAs'] + curves['CRs']) > 40]

,curveID,Ref Amp,Ref PW,Active Channels,Return Channels,Monkey,Date,FAs,CRs,n_Amp,n_PW,Days
0,1.0,0,0,240,4,U,2016-10-05,24.0,245,8,1,9
1,4.0,0,0,240,2,U,2016-10-06,17.0,280,8,1,10
2,5.0,0,0,15,128,U,2016-10-07,8.0,86,8,1,11
3,6.0,0,0,15,16,U,2016-10-07,5.0,199,8,1,11
4,7.0,0,0,15,240,U,2016-10-10,25.0,320,8,1,14
...,...,...,...,...,...,...,...,...,...,...,...,...
888,1117.0,0,0,8,128,Y,2016-09-12,83.0,326,2,1,11
889,1118.0,0,0,15,128,Y,2016-09-20,185.0,236,15,1,19
890,1119.0,0,0,15,64,Y,2016-09-21,92.0,193,15,1,20
891,1120.0,0,0,15,32,Y,2016-09-22,103.0,193,8,1,21


In [81]:
curves.to_csv('data/tidy_data/curves.csv',index=False)
points.to_csv('data/tidy_data/points.csv',index=False)

In [14]:
def fit_curves(group):
    data = group[['x','HR','n']]
    group[]
curves_points.groupby(['curveID', 'base']).apply()

RangeIndex(start=0, stop=60916, step=1)